# 16.C21A/18.C21A Problem Set #3
Due Thursday 23 April 2026 at 11:59 pm ET.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Problem 1: Finite-Volume Conservation and Numerical Fluxes (5+5+5)

Consider the scalar conservation law $u_t + f(u)_x = 0$ in finite-volume form.

A key property of a conservative scheme is the **telescoping sum**: if we sum the update over cells $j = j_L, \ldots, j_R$:

$$\sum_{j=j_L}^{j_R} U_j^{n+1} = \sum_{j=j_L}^{j_R} U_j^n - \frac{\Delta t}{\Delta x}\bigl(F_{j_R+1/2} - F_{j_L-1/2}\bigr),$$

only the boundary fluxes remain. This property is called **discrete conservation**.

**1(a)** Telescoping property proof (5 pts)

For the general conservative scheme:
$$U_j^{n+1} = U_j^n - \frac{\Delta t}{\Delta x}\bigl(F_{j+1/2}^n - F_{j-1/2}^n\bigr),$$

show that summing over any contiguous block of cells $j = j_L, \ldots, j_R$ yields a **telescoping sum** where only the boundary fluxes survive. (This is the discrete analog of the integral conservation law.)

*Hint:* Just sum over $j$; the interior fluxes cancel pairwise.

**1(b)** Lax-Friedrichs and Lax-Wendroff fluxes (5 pts)

The **Lax-Friedrichs** (LF) numerical flux is:
$$F_{j+1/2}^{\text{LF}} = \frac{1}{2}\bigl[f(U_j) + f(U_{j+1})\bigr] - \frac{\alpha}{2}(U_{j+1} - U_j),$$
where $\alpha = \max_u |f'(u)|$ (global wave speed bound).

The **Lax-Wendroff** (LW) numerical flux (Richtmyer two-step version) is:
$$U_{j+1/2}^{n+1/2} = \frac{1}{2}(U_j + U_{j+1}) - \frac{\Delta t}{2\Delta x}\bigl[f(U_{j+1}) - f(U_j)\bigr], \qquad F_{j+1/2}^{\text{LW}} = f\!\left(U_{j+1/2}^{n+1/2}\right).$$

1. For $f(u) = u^2/2$ (Burgers' equation), write the complete LF update formula, $U_j^{n+1} = \ldots$, explicitly.
2. Show that the Lax-Friedrichs scheme is equivalent to applying Lax-Wendroff to a **modified** equation with added numerical viscosity $\mu_{\text{num}} = \frac{\alpha\Delta x}{2}\left(1 - \text{CFL}\right)$, where $\text{CFL} = \alpha\Delta t/\Delta x$. What happens at CFL $= 1$?

**1(c)** Numerical experiment: solving Burgers’ equation with Lax-Friedrichs (5 pts)

Solve $u_t + (u^2/2)_x = 0$ on $[-1, 1]$ with periodic BCs using the single-step forward-in-time update $U_j^{n+1} = U_j^n - \frac{\Delta t}{\Delta x}(F_{j+1/2}^n - F_{j-1/2}^n)$ with the Lax-Friedrichs flux. Use the initial condition $u_0(x) = \sin(\pi x)$.

1. Implement the Lax-Friedrichs numerical flux (formula in part (b)).
2. Run with $N_x = 200$, CFL $= 0.5$. Plot the solution at $t = 0,\; 0.2,\; 1/\pi$ (shock formation time), and $t = 0.5$ (well after the shock forms).
3. Verify discrete conservation: track $M^n = \Delta x \sum_j U_j^n$ over time and confirm that it is preserved to machine precision.
4. Run at CFL $= 0.9$ and compare the shock profile at $t = 0.5$ to the CFL $= 0.5$ result. Which is sharper? Explain in terms of the numerical viscosity $\mu_{\text{num}}$ from part (b).

In [ ]:
# Problem 1(c): Burgers' equation — Lax-Friedrichs

Nx = 200
x  = np.linspace(-1, 1, Nx, endpoint=False)
dx = x[1] - x[0]
CFL = 0.5

def burgers_flux(u):
    return 0.5 * u**2

def lax_friedrichs_step(U, dx, dt):
    """One LF step for Burgers'. Returns U^{n+1}."""
    alpha = np.max(np.abs(U))  # global wave speed bound
    fU  = burgers_flux(U)
    Up1 = np.roll(U, -1)  # U_{j+1} with periodic BCs
    # Numerical flux at j+1/2
    F_right = 0.5 * (fU + burgers_flux(Up1)) - 0.5 * alpha * (Up1 - U)
    return U - (dt / dx) * (F_right - np.roll(F_right, 1))

def run_lf(U0, dx, CFL, T_final):
    """Advance to T_final with adaptive dt. Returns (U, times, masses)."""
    U = U0.copy()
    t = 0.0
    times, masses = [0.0], [dx * np.sum(U)]
    while t < T_final - 1e-14:
        alpha = max(np.max(np.abs(U)), 1e-10)
        dt = CFL * dx / alpha
        if t + dt > T_final:
            dt = T_final - t
        U = lax_friedrichs_step(U, dx, dt)
        t += dt
        times.append(t)
        masses.append(dx * np.sum(U))
    return U, np.array(times), np.array(masses)

U0 = np.sin(np.pi * x)

# TODO: Plot at t = 0, 0.2, 1/pi, 0.5
# TODO: Check mass conservation
# TODO: Compare CFL = 0.5 vs CFL = 0.9 at t = 0.5


In [ ]:
# Problem 1(c): Julia version

Nx = 200
x  = range(-1, stop=1-2/Nx, length=Nx) |> collect
dx = x[2] - x[1]

burgers_flux(u) = 0.5 .* u.^2

function lax_friedrichs_step(U, dx, dt)
    alpha = maximum(abs.(U))
    fU  = burgers_flux(U)
    Up1 = circshift(U, -1)  # U_{j+1}, periodic
    F_right = 0.5 .* (fU .+ burgers_flux(Up1)) .- 0.5 * alpha .* (Up1 .- U)
    return U .- (dt / dx) .* (F_right .- circshift(F_right, 1))
end

function run_lf(U0, dx, CFL, T_final)
    U = copy(U0)
    t = 0.0
    times  = [0.0]
    masses = [dx * sum(U)]
    while t < T_final - 1e-14
        alpha = max(maximum(abs.(U)), 1e-10)
        dt = CFL * dx / alpha
        if t + dt > T_final
            dt = T_final - t
        end
        U = lax_friedrichs_step(U, dx, dt)
        t += dt
        push!(times, t)
        push!(masses, dx * sum(U))
    end
    return U, times, masses
end

U0 = sin.(π .* x)

# TODO: Plot at t = 0, 0.2, 1/π, 0.5
# TODO: Check mass conservation
# TODO: Compare CFL = 0.5 vs CFL = 0.9 at t = 0.5


## Problem 2: Method of Weighted Residuals (5+10+10)

Consider the 1D diffusion problem
$$
\frac{d^2 T}{dx^2} = \sin(2\pi x), \qquad T(0)=T(1)=1.
$$


**2(a)** Determine the exact analytical solution of the boundary-value problem.


**2(b)** Assume that the approximate solution has the form
$$
\widetilde{T}(x)=1+a_1x(1-x)+a_2x^2(1-x)+a_3x^3(1-x),
$$
where $a_1,a_2,a_3 \in \mathbb{R}$ are unknown coefficients.

Apply the Galerkin method of weighted residuals and derive the resulting $3\times 3$ linear system for $(a_1,a_2,a_3)^\top$.


**2(c)** Determine whether the $3\times 3$ system has a unique solution. If it does, compute $(a_1,a_2,a_3)$ and compare $\widetilde{T}(x)$ with the exact solution in a plot.


In [ ]:
# Problem 2(c): solve the Galerkin system and compare with the exact solution

pi = np.pi
x = np.linspace(0.0, 1.0, 400)

def T_exact(x):
    return 1.0 - np.sin(2.0 * pi * x) / (4.0 * pi**2)

# Fill this matrix and RHS from your derivation in part (b)
A = np.array([
    [-1/3,  -1/6,  -1/10],
    [-1/6,  -2/15, -1/10],
    [-1/10, -1/10, -3/35],
], dtype=float)
b = np.array([
    0.0,
    -3.0 / (4.0 * pi**3),
    -3.0 / (4.0 * pi**3),
], dtype=float)

def T_tilde(x, a):
    a1, a2, a3 = a
    return 1.0 + a1 * x * (1 - x) + a2 * x**2 * (1 - x) + a3 * x**3 * (1 - x)

# TODO: solve A a = b, report the coefficients, and plot T_tilde vs T_exact.


In [ ]:
# Problem 2(c): Julia version
using LinearAlgebra, Plots

A = [-1/3  -1/6  -1/10;
     -1/6  -2/15 -1/10;
     -1/10 -1/10 -3/35]
b = [0.0, -3/(4π^3), -3/(4π^3)]

T_exact(x) = 1.0 - sin(2π*x) / (4π^2)
T_tilde(x, a) = 1.0 + a[1]*x*(1-x) + a[2]*x^2*(1-x) + a[3]*x^3*(1-x)

# TODO: solve A a = b, report the coefficients, and plot T_tilde vs T_exact.


## Problem 3: Finite Element Method for 1D Diffusion with Variable Conductivity (10+5+5+10)

Consider the steady diffusion problem
$$
\left(k T_x\right)_x = -q,
$$
on the interval $[-1,1]$, with
$$
q(x)=50e^x, \qquad T(-1)=T(1)=100,
$$
and a linearly varying conductivity
$$
k(x)=k_0+k_1x.
$$
Use linear finite elements and the nodal basis.


**3(a)** Starting from the Galerkin residual
$$
R_j(\widetilde{T}) = \left[\phi_j k\widetilde{T}_x\right]_{-L/2}^{L/2} - \int_{-L/2}^{L/2} \phi_{j,x}\,k\widetilde{T}_x\,dx + \int_{-L/2}^{L/2}\phi_j q\,dx,
$$
analytically evaluate the term
$$
\int_{-L/2}^{L/2} \phi_{j,x}\,k\widetilde{T}_x\,dx
$$
for $k(x)=k_0+k_1x$.


**3(b)** For interior nodes, write the corresponding nonzero entries of the stiffness matrix.


**3(c)** Complete the 1D diffusion FEM code below to handle the linearly varying conductivity.


**3(d)** Using $k_0=1$ and $k_1=0.1$, solve the problem for $\texttt{nElem}=5$, $10$, and $20$. Plot the three numerical solutions on the same figure.


In [ ]:
# Problem 3: linear FEM for variable conductivity

def assemble_variable_k_system(nElem, k0=1.0, k1=0.1):
    x = np.linspace(-1.0, 1.0, nElem + 1)
    K = np.zeros((nElem + 1, nElem + 1))
    f = np.zeros(nElem + 1)

    for e in range(nElem):
        a, b = x[e], x[e + 1]
        h = b - a

        # TODO: insert the exact local stiffness matrix for k(x)=k0+k1*x
        # ke = ...

        # TODO: insert the element load vector for q(x)=50*exp(x)
        # fe = ...

        idx = [e, e + 1]
        # TODO: scatter ke and fe into the global K and f arrays

    T = np.zeros(nElem + 1)
    T[0] = T[-1] = 100.0

    # TODO: apply Dirichlet boundary conditions and solve for interior nodes
    # rhs = ...
    # T[1:-1] = np.linalg.solve(K[1:-1, 1:-1], rhs)
    return x, T

for nElem in [5, 10, 20]:
    x, T = assemble_variable_k_system(nElem)
    # TODO: plot each mesh solution on the same axes


In [ ]:
# Problem 3(d): Julia version — linear FEM for variable conductivity
using LinearAlgebra, Plots

function assemble_variable_k(nElem; k0=1.0, k1=0.1)
    x = range(-1.0, 1.0, length=nElem + 1) |> collect
    K = zeros(nElem + 1, nElem + 1)
    f = zeros(nElem + 1)

    for e in 1:nElem
        a, b = x[e], x[e+1]
        h = b - a

        # TODO: compute the local stiffness matrix for k(x) = k0 + k1*x
        # ke = ...

        # TODO: compute the element load vector for q(x) = 50*exp(x)
        # fe = ...

        # TODO: scatter ke and fe into K and f
    end

    T = zeros(nElem + 1)
    T[1] = 100.0;  T[end] = 100.0

    # TODO: apply Dirichlet BCs and solve for interior nodes
    return x, T
end

for nElem in [5, 10, 20]
    x, T = assemble_variable_k(nElem)
    # TODO: plot each mesh solution on the same axes
end